### **Package**

In [ ]:
import subprocess
import sys
import os
import warnings
import importlib.util

print(sys.version)

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*Assigning the 'data' attribute.*")
warnings.filterwarnings("ignore", module="paramz.*")
warnings.filterwarnings("ignore", message=".*load_learner.*pickle.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is not None:
        print(f"{import_name} is already installed.")
    else:
        print(f"{import_name} is not installed. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
        print(f"{import_name} installed.")

# Upgrade jupyter_client to reduce datetime.utcnow warning
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U", "jupyter_client>=8.5.0"
])

ensure_package("pymoo")
ensure_package("GPy")
ensure_package("pythermalcomfort")
ensure_package("plotly")

In [ ]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from pathlib import Path
from pymoo.core.problem import Problem
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pathlib import Path

# Optimization algorithm
from pymoo.optimize import minimize
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.termination import get_termination
from pymoo.core.survival import Survival
from pymoo.core.mutation import Mutation
from pymoo.core.sampling import Sampling
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from pymoo.operators.survival.rank_and_crowding.metrics import get_crowding_function
from pymoo.util.randomized_argsort import randomized_argsort

# Surrogate model
import GPy

# Thermal comfort model
from pythermalcomfort.models import pmv_ppd_ashrae
from pythermalcomfort.utilities import v_relative

# Metrics
from pymoo.indicators.hv import HV


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('xxx')
DATA_PATH = PROJECT_DIR / 'Dataset' / 'data_office_1.csv'

print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_PATH:', DATA_PATH)

### **Class and Function**

###### Plot and Metrics

In [ ]:
# Plot: 2 Objs and pareto front
def plot_obj_2d(F, xlim=None, ylim=None):
    n_obj = F.shape[1]
    if n_obj == 2:
        nds = NonDominatedSorting()
        front_idx = nds.do(F, only_non_dominated_front=True)

        pareto_F = F[front_idx]
        non_pareto_F = np.delete(F, front_idx, axis=0)

        fig = go.Figure(
            data=go.Scatter(
                x=F[:, 0],
                y=F[:, 1],
                mode='markers',
                name='Objective Values',
                marker=dict(size=6, color='#87CEEB', opacity=0.7)
            )
        )
        fig.add_trace(go.Scatter(
            x=pareto_F[:, 0],
            y=pareto_F[:, 1],
            mode='markers',
            name='Pareto Front',
            marker=dict(size=7, color='#FF7F0E', opacity=0.9, symbol='diamond')
        ))
        layout = dict(xaxis_title='f1 - cost', yaxis_title='f2 - thermal comfort', width=600, height=600)
        if xlim is not None:
            layout['xaxis'] = dict(range=list(xlim))
        if ylim is not None:
            layout['yaxis'] = dict(range=list(ylim))
        fig.update_layout(**layout)
        fig.show()


def mean_std(arr):
    return np.mean(arr), np.std(arr)


def format_mean_std(mean, std, digits=2):
    if mean == 0:
        return f"(0 ± {std:.{digits}g})"
    exp = int(np.floor(np.log10(abs(mean))))
    scale = 10**exp
    mean_scaled = mean / scale
    std_scaled = std / scale
    return f"({mean_scaled:.{digits}f}±{std_scaled:.{digits}f}) e{exp:+d}"


###### Class: Surrogate model

In [ ]:
# Model: Kriging_RBF
class Kriging_RBF:
    def __init__(self):
        self.model = None
        self.x_scaler = StandardScaler()

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        X_scaled = self.x_scaler.fit_transform(X)
        dim = X_scaled.shape[1]

        kernel = GPy.kern.RBF(input_dim=dim, ARD=True)
        self.model = GPy.models.GPRegression(X_scaled, y, kernel, normalizer=True)
        self.model.optimize(messages=False)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        X_scaled = self.x_scaler.transform(X)
        y_mean, y_var = self.model.predict(X_scaled, include_likelihood=True)
        y_std = np.sqrt(np.maximum(y_var, 0))
        return y_mean.flatten(), y_std.flatten()


In [ ]:
# Model: Kriging_RBF
class Kriging_RBF_small_noise:
    def __init__(self):
        self.model = None
        self.x_scaler = StandardScaler()

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        X_scaled = self.x_scaler.fit_transform(X)
        dim = X_scaled.shape[1]

        kernel = GPy.kern.RBF(input_dim=dim, ARD=True)
        self.model = GPy.models.GPRegression(X_scaled, y, kernel, normalizer=True)
        self.model.Gaussian_noise.variance = 1e-6
        self.model.Gaussian_noise.variance.fix()
        self.model.optimize(messages=False)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        X_scaled = self.x_scaler.transform(X)
        y_mean, y_var = self.model.predict(X_scaled, include_likelihood=False)
        y_std = np.sqrt(np.maximum(y_var, 0))
        return y_mean.flatten(), y_std.flatten()


###### Define Problem

In [ ]:
# Problem
class Flexible_Wall_Problem(Problem):
    def __init__(self, model_f1, n_var, n_obj, xl, xu,
                 fixed_wall_list, room_types, required_num_meeting_room,
                 space_width, room_area_min, initial_occ_list, total_occ,
                 occ_min_area, th_zone_list, outdoor_t, outdoor_rh,
                 wind_speed, solar_radiation, clo, met, v,
                 use_surrogate):

        self.model_f1 = model_f1
        self.use_surrogate = use_surrogate
        self.fixed_wall_list = np.asarray(fixed_wall_list, dtype=float)
        self.room_types = np.asarray(room_types)
        self.required_num_meeting_room = required_num_meeting_room
        self.space_width = space_width
        self.room_area_min = room_area_min
        self.initial_occ_list = np.asarray(initial_occ_list, dtype=float)
        self.total_occ = total_occ
        self.occ_min_area = occ_min_area
        self.th_zone_list = np.asarray(th_zone_list, dtype=float)
        self.outdoor_t = outdoor_t
        self.outdoor_rh = outdoor_rh
        self.wind_speed = wind_speed
        self.solar_radiation = solar_radiation
        self.clo = clo
        self.met = met
        self.v = v

        n_rooms = n_var - 1
        n_fixed = int(np.sum(self.fixed_wall_list >= 0))
        n_constr = n_rooms + n_fixed + 1 + n_rooms + 1 + n_rooms

        super().__init__(n_var=n_var, n_obj=n_obj, xl=xl, xu=xu, n_constr=n_constr)

    def _evaluate(self, X, out, *args, **kwargs):
        X_eval = np.asarray(X, dtype=float).copy()
        fixed_indices = np.where(self.fixed_wall_list >= 0)[0]
        X_eval[:, fixed_indices] = self.fixed_wall_list[fixed_indices]

        n_solutions, n_vars = X_eval.shape
        n_rooms = n_vars - 1

        f1_mean_list = []
        f1_std_list = []
        f2_list = []
        occ_record_list = []
        area_occ_record = np.zeros((n_solutions, n_rooms))
        room_area_record = np.zeros((n_solutions, n_rooms))

        for i, single_solution in enumerate(X_eval):
            room_area_list = np.diff(single_solution) * self.space_width
            room_area_list[room_area_list < 0] = 0

            room_occ_list = self.occ_allocate(room_area_list)
            occ_record_list.append(abs(room_occ_list.sum() - self.total_occ))
            area_occ_record[i, :] = room_occ_list * self.occ_min_area - room_area_list
            room_area_record[i, :] = room_area_list - self.room_area_min

            room_t_list, room_rh_list = self.update_room_t_and_rh(single_solution)

            features_f1 = np.column_stack((
                room_occ_list,
                room_t_list,
                room_rh_list,
                np.full(n_rooms, self.outdoor_t, dtype=float),
                np.full(n_rooms, self.outdoor_rh, dtype=float),
                np.full(n_rooms, self.wind_speed, dtype=float),
                np.full(n_rooms, self.solar_radiation, dtype=float)))

            features_f2 = np.column_stack((
                room_t_list,
                room_t_list,
                np.full(n_rooms, self.v, dtype=float),
                room_rh_list,
                np.full(n_rooms, self.met, dtype=float),
                np.full(n_rooms, self.clo, dtype=float)))

            y1_mean, y1_std = self.model_f1.predict(features_f1)
            weights = room_area_list / np.sum(room_area_list)
            f1_mean = float(np.sum(y1_mean * weights))
            f1_std = float(np.sqrt(np.sum((y1_std * weights) ** 2)))
            f2_val = self.calculate_pmv(features_f2, room_occ_list)

            f1_mean_list.append(f1_mean)
            f1_std_list.append(f1_std)
            f2_list.append(f2_val)

        f1_mean_list = np.asarray(f1_mean_list, dtype=float)
        f1_std_list = np.asarray(f1_std_list, dtype=float)
        f2_list = np.asarray(f2_list, dtype=float)

        out["F"] = np.column_stack([f1_mean_list, f2_list])
        out["std"] = np.column_stack([f1_std_list, np.zeros_like(f2_list)])

        constraints_X = np.diff(X_eval)
        constraints_fixed_walls = np.column_stack([
            np.abs(X_eval[:, i] - self.fixed_wall_list[i])
            for i in range(len(self.fixed_wall_list))
            if self.fixed_wall_list[i] >= 0
        ])
        constraints_total_occ = np.asarray(occ_record_list, dtype=float).reshape(-1, 1)
        constraints_occ_area = area_occ_record
        num_meeting_room = np.count_nonzero(self.room_types == 'meeting_room')
        constraints_num_meeting_room = np.full((X_eval.shape[0], 1),
                                               self.required_num_meeting_room - num_meeting_room)
        constraints_room_area = room_area_record

        out["G"] = np.column_stack([
            -constraints_X,
            constraints_fixed_walls,
            constraints_total_occ,
            constraints_occ_area,
            constraints_num_meeting_room,
            -constraints_room_area])

    def update_room_t_and_rh(self, single_solution):
        zone_start = self.th_zone_list[:, 0]
        zone_end = self.th_zone_list[:, 1]
        zone_t = self.th_zone_list[:, 2]
        zone_rh = self.th_zone_list[:, 3]

        left_boundaries = single_solution[:-1]
        right_boundaries = single_solution[1:]
        room_lengths = right_boundaries - left_boundaries

        room_t_list = np.zeros(len(room_lengths))
        room_rh_list = np.zeros(len(room_lengths))

        penalties_mask = room_lengths <= 0
        room_t_list[penalties_mask] = 32.0
        room_rh_list[penalties_mask] = 100.0

        valid_mask = room_lengths > 0
        overlap_start = np.maximum(left_boundaries[valid_mask, np.newaxis], zone_start)
        overlap_end = np.minimum(right_boundaries[valid_mask, np.newaxis], zone_end)
        overlap_length = np.clip(overlap_end - overlap_start, 0, None)
        overlap_ratios = overlap_length / room_lengths[valid_mask, np.newaxis]

        room_t_list[valid_mask] = np.sum(overlap_ratios * zone_t, axis=1)
        room_rh_list[valid_mask] = np.sum(overlap_ratios * zone_rh, axis=1)
        return room_t_list, room_rh_list

    def occ_allocate(self, room_area_list):
        office_area_list = room_area_list * (self.room_types == 'office')
        max_possible_occ = np.floor(office_area_list / self.occ_min_area).astype(int)
        excess_occupancy = np.maximum(self.initial_occ_list - max_possible_occ, 0)
        occ_remain = int(np.sum(excess_occupancy))
        new_occ_list = self.initial_occ_list - excess_occupancy

        for j, single_room_area in enumerate(office_area_list):
            max_occ_addable = int((single_room_area // self.occ_min_area) - self.initial_occ_list[j])
            if max_occ_addable >= 1 and occ_remain > 0:
                occ_added = min(max_occ_addable, occ_remain)
                new_occ_list[j] += occ_added
                occ_remain -= occ_added
            if occ_remain == 0:
                break
        return new_occ_list

    def calculate_pmv(self, features_batch, room_occ_list):
        pmv_array = []
        for feature in features_batch:
            indoor_t, tr, v, indoor_rh, met, clo = feature
            v_r = v_relative(v=v, met=met)
            result = pmv_ppd_ashrae(tdb=indoor_t, tr=tr, vr=v_r, rh=indoor_rh,
                                    met=met, clo=clo, model="55-2023")
            pmv_array.append(result["pmv"])

        pmv_array = np.asarray(pmv_array, dtype=float)
        room_occ_list = np.asarray(room_occ_list, dtype=float)
        if np.sum(room_occ_list) <= 0:
            return float(np.mean(np.abs(pmv_array)))
        return float(np.sum(np.abs(pmv_array) * room_occ_list) / np.sum(room_occ_list))


###### Class: Survival_standard

In [ ]:
class Survival_standard(Survival):

    def __init__(self, nds=None, crowding_func="cd"):
        crowding_func_ = get_crowding_function(crowding_func)
        super().__init__(filter_infeasible=True)
        self.nds = nds if nds is not None else NonDominatedSorting()
        self.crowding_func = crowding_func_


    def _do(self,
            problem,
            pop,
            *args,
            random_state=None,
            n_survive=None,
            **kwargs):

        F = pop.get("F").astype(float, copy=False)

        survivors = []

        fronts = self.nds.do(F, n_stop_if_ranked=n_survive)

        for k, front in enumerate(fronts):

            I = np.arange(len(front))

            if len(survivors) + len(I) > n_survive:
                n_remove = len(survivors) + len(front) - n_survive
                crowding_of_front = \
                    self.crowding_func.do(
                        F[front, :],
                        n_remove=n_remove
                    )

                I = randomized_argsort(crowding_of_front, order='descending', method='numpy', random_state=random_state)
                I = I[:-n_remove]

            else:
                crowding_of_front = \
                    self.crowding_func.do(
                        F[front, :],
                        n_remove=0
                    )

            for j, i in enumerate(front):
                pop[i].set("rank", k)
                pop[i].set("crowding", crowding_of_front[j])
            survivors.extend(front[I])

        return pop[survivors]


###### Class: Survival_dual_ranking

In [ ]:
class Survival_dual_ranking(Survival):

    def __init__(self, nds=None, crowding_func="cd", alpha_f1=1, alpha_f2=1):
        crowding_func_ = get_crowding_function(crowding_func)
        super().__init__(filter_infeasible=True)
        self.nds = nds if nds is not None else NonDominatedSorting()
        self.crowding_func = crowding_func_
        self.alpha_f1 = alpha_f1
        self.alpha_f2 = alpha_f2


    def _do(self,problem,pop,*args,random_state=None,n_survive=None,**kwargs):
        F = pop.get("F").astype(float, copy=False)
        F_std = pop.get("std").astype(float, copy=False)

        #============= F_upper =====================
        alphas = np.array([self.alpha_f1, self.alpha_f2])
        F_upper = F + alphas * F_std
        F_hybrid = np.concatenate([F, F_upper], axis=1)
        #===========================================

        # ====== NonDominatedSorting ============
        fronts_hybrid = NonDominatedSorting().do(F_hybrid)
        #==========================================

        survivors = []
        for k, front in enumerate(fronts_hybrid):

            I = np.arange(len(front))

            if len(survivors) + len(I) > n_survive:
                n_remove = len(survivors) + len(front) - n_survive
                crowding_of_front = \
                    self.crowding_func.do(
                        F[front, :],
                        n_remove=n_remove
                    )

                I = randomized_argsort(crowding_of_front, order='descending', method='numpy', random_state=random_state)
                I = I[:-n_remove]

            else:
                crowding_of_front = \
                    self.crowding_func.do(
                        F[front, :],
                        n_remove=0
                    )

            for j, i in enumerate(front):
                pop[i].set("rank", k)
                pop[i].set("crowding", crowding_of_front[j])
            survivors.extend(front[I])

        return pop[survivors]


###### Class: Sampling and Mutation

In [ ]:
class FixedWallMutation(Mutation):
    def __init__(self, prob, eta, fixed_wall_list):
        super().__init__()
        self.prob = prob
        self.eta = eta
        self.fixed_wall_list = np.asarray(fixed_wall_list, dtype=float)

    def _do(self, problem, X, **kwargs):
        X = np.asarray(X, dtype=float).copy()
        movable_indices = np.where(self.fixed_wall_list < 0)[0]
        mutation_mask = np.random.random(X[:, movable_indices].shape) < self.prob
        random_num = np.random.normal(0, self.eta, size=mutation_mask.shape)
        X[:, movable_indices] += mutation_mask * random_num
        X = np.clip(X, problem.xl, problem.xu)
        X = np.sort(X, axis=1)
        fixed_indices = np.where(self.fixed_wall_list >= 0)[0]
        X[:, fixed_indices] = self.fixed_wall_list[fixed_indices]
        return X


class FromArraySampling(Sampling):
    def __init__(self, initial_population):
        super().__init__()
        self.initial_population = np.asarray(initial_population, dtype=float)

    def _do(self, problem, n_samples, **kwargs):
        n_initial = self.initial_population.shape[0]
        if n_initial < n_samples:
            extra = n_samples - n_initial
            random_samples = np.random.uniform(problem.xl, problem.xu, (extra, problem.n_var))
            return np.vstack([self.initial_population, random_samples])
        else:
            return self.initial_population[:n_samples]


def make_initial_wall_population(initial_solution, fixed_wall_list, pop_size, min_room_length=3.0, seed=42):
    population = np.tile(initial_solution, (pop_size, 1)).astype(float)
    if pop_size <= 1:
        return population

    rng = np.random.default_rng(seed)
    total_length = float(initial_solution[-1] - initial_solution[0])
    n_rooms = len(initial_solution) - 1
    free_length = total_length - n_rooms * min_room_length
    fixed_indices = np.where(fixed_wall_list >= 0)[0]

    for i in range(1, pop_size):
        room_lengths = min_room_length + rng.dirichlet(np.ones(n_rooms)) * free_length
        candidate = np.concatenate([[initial_solution[0]], initial_solution[0] + np.cumsum(room_lengths)])
        candidate[fixed_indices] = fixed_wall_list[fixed_indices]
        population[i] = candidate
    return population


### **Main**

###### 1. Initial settings

In [ ]:
FEATURE_COLUMNS = [
    'occupant_count [number]',
    'air_temperature [Celsius]',
    'indoor_relative_humidity [%]',
    'dry_bulb_temp [Celsius]',
    'outdoor_relative_humidity [%]',
    'wind_speed [m/s]',
    'global_horizontal_solar_radiation [W/m2]']

ENERGY_COLUMNS = [
    'ceiling_fan_energy [kWh]',
    'chilled_water_energy [kWh]',
    'ahu_fan_energy [kWh]']


data = pd.read_csv(DATA_PATH)
data['date'] = pd.to_datetime(data['timestamp'])
data.set_index('date', inplace=True)

# Keep the same selected time intervals as 1_QR.ipynb / 1_ML_AutoGluon.ipynb
data_temp1 = data.loc['2021-09-07':'2021-09-08'].copy()
data_temp2 = data.loc['2021-09-10':'2021-09-10'].copy()
data_temp3 = data.loc['2021-09-13':'2021-09-17'].copy()
data_temp4 = data.loc['2021-09-20':'2021-09-23'].copy()

data_temp2.index = data_temp2.index.map(lambda x: x.replace(day=9))
data_temp3.index = data_temp3.index.map(lambda x: x + pd.DateOffset(days=-3))
data_temp4.index = data_temp4.index.map(lambda x: x + pd.DateOffset(days=-5))

data_combined = pd.concat([data_temp1, data_temp2, data_temp3, data_temp4]).sort_index()
print('data_combined.shape:', data_combined.shape)

X = data_combined[FEATURE_COLUMNS].copy()
y_sum = data_combined[ENERGY_COLUMNS].sum(axis=1).rename('total_energy')
repair_time = pd.Timestamp('2021-09-10 17:25:00')
if repair_time in y_sum.index:
    y_sum.loc[repair_time] = (y_sum.loc['2021-09-10 17:20:00'] + y_sum.loc['2021-09-10 17:30:00']) / 2

xy = pd.concat([X, y_sum], axis=1).dropna()
X = xy[FEATURE_COLUMNS]
Y_sum = xy['total_energy']

train_size = 288 * 10
X_train_full, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train_full, y_test_f1 = Y_sum.iloc[:train_size], Y_sum.iloc[train_size:]

val_size = int(round(len(X_train_full) * 0.2))
X_train = X_train_full.iloc[:-val_size]
y_train_f1 = y_train_full.iloc[:-val_size]
X_val = X_train_full.iloc[-val_size:]
y_val_f1 = y_train_full.iloc[-val_size:]

# GPR is exact and cubic in training size. Use a subset by default for practical Colab runtime.
max_train = 0   # set 0 to use all rows
train_seed = 42
if max_train > 0 and len(X_train) > max_train:
    rng = np.random.default_rng(train_seed)
    idx = np.sort(rng.choice(len(X_train), size=max_train, replace=False))
    X_train_model = X_train.iloc[idx].to_numpy(dtype=float)
    y_train_model_f1 = y_train_f1.iloc[idx].to_numpy(dtype=float)
else:
    X_train_model = X_train.to_numpy(dtype=float)
    y_train_model_f1 = y_train_f1.to_numpy(dtype=float)

X_val_np = X_val.to_numpy(dtype=float)
y_val_np_f1 = y_val_f1.to_numpy(dtype=float)
X_test_np = X_test.to_numpy(dtype=float)
y_test_np_f1 = y_test_f1.to_numpy(dtype=float)

print('X_train_model shape:', X_train_model.shape)
print('X_val shape:', X_val_np.shape)
print('X_test shape:', X_test_np.shape)

# Flexible-wall optimization settings
problem_name = 'flexible_wall_office'
n_var = 7
n_obj = 2
n_gen = 50
pop_size = 100
seed = 1

position_first_wall = 0.0
position_last_wall = 60.0
initial_solution = np.array([0.0, 10.0, 20.0, 30.0, 40.0, 50.0, 60.0])
fixed_wall_list = np.array([position_first_wall, -1, -1, -1, -1, -1, position_last_wall])

room_types_list = np.array(['office', 'office', 'office', 'office', 'office', 'meeting_room'])
required_num_meeting_room = 1
space_width = 2.0
room_length_min = 3.0
room_area_min = room_length_min * space_width
initial_occ_list = np.array([3, 3, 3, 2, 2, 0])
total_occ = int(np.sum(initial_occ_list))
occ_min_area = 6.0

outdoor_t = 27.5
outdoor_rh = 85.0
wind_speed = 1.0
solar_radiation = 650.0
clo_default = 0.5
met_default = 1.0
v_default = 0.1

th_zone_list = np.array([
    [0, 10, 22.5, 85],
    [10, 20, 23.0, 82],
    [20, 30, 23.5, 80],
    [30, 40, 22.9, 86],
    [40, 50, 22.2, 89],
    [50, 60, 22.0, 85]
], dtype=float)

xl = np.zeros(n_var)
xu = np.ones(n_var) * position_last_wall
initial_population = make_initial_wall_population(initial_solution, fixed_wall_list, pop_size,
                                                  min_room_length=room_length_min, seed=42)

ref_point = np.array([0.5, 1.2])
hv = HV(ref_point=ref_point)
print('\nProblem name:', problem_name)
print('Var:', n_var)
print('Obj:', n_obj)
print('HV Reference point:', ref_point)


###### 2. Surrogate model training

In [ ]:
# Kriging
def model(X_train, y_train_f1, X_test, y_test_f1, model_name):
    if model_name == 'Kriging_RBF':
        model_kriging_f1 = Kriging_RBF()
    elif model_name == 'Kriging_RBF_small_noise':
        model_kriging_f1 = Kriging_RBF_small_noise()
    else:
        raise ValueError(f'Unknown model_name: {model_name}')

    print('model_name: ', model_name)
    model_kriging_f1.fit(X_train, y_train_f1)

    # Predict
    mean_f1, std_f1 = model_kriging_f1.predict(X_test)
    pred_mean = mean_f1.reshape(-1, 1)
    pred_std = std_f1.reshape(-1, 1)

    # Metrics
    mse_kriging = mean_squared_error(y_test_f1, mean_f1)
    mae_kriging = mean_absolute_error(y_test_f1, mean_f1)
    r2_kriging = r2_score(y_test_f1, mean_f1)
    print(f"Kriging(RBF) MSE: {mse_kriging:.2e}")
    print(f"Kriging(RBF) MAE: {mae_kriging:.2e}")
    print(f"Kriging(RBF) R2: {r2_kriging:.3f}\n")

    print("f1 lengthscale:", model_kriging_f1.model.kern.lengthscale.values)
    print("f1 kernel variance:", model_kriging_f1.model.kern.variance.values)
    print("f1 noise:", model_kriging_f1.model.Gaussian_noise.variance.values)

    print('pred_mean\n', pred_mean[0:5])
    print('pred_std\n', pred_std[0:5])
    print('Max pred_std\n', np.max(pred_std, axis=0))

    return model_kriging_f1


In [ ]:
model_kriging_f1 = model(X_train_model, y_train_model_f1, X_test_np, y_test_np_f1, 'Kriging_RBF_small_noise')


###### 2.1 CICP & Z score

In [ ]:
mean_f1, std_f1 = model_kriging_f1.predict(X_test_np)
pred_mean = mean_f1.reshape(-1, 1)
pred_std = std_f1.reshape(-1, 1)
y_test_arr = y_test_np_f1.reshape(-1, 1)

###### CICP
"""
Interval    Coverage
±1.282σ      80.00%
±1.645σ      90.00%
±1.960σ      95.00%
"""
def coverage(y_test, pred_mean, pred_std, k=1.0):
    err = np.abs(y_test - pred_mean)
    inside = err <= k * pred_std
    per_dim = inside.mean(axis=0)
    overall = inside.mean()
    return per_dim, overall

print('Coverage')
for k in [1.645]:
    per_dim, overall = coverage(y_test_arr, pred_mean, pred_std, k=k)
    print(f"k={k}: per_dim={per_dim*100}%, overall={overall*100:.1f}%")

###### Z score
z = (y_test_arr - pred_mean) / pred_std
plt.figure(figsize=(5, 4))
plt.hist(z[:, 0], bins=30)
plt.title("Z distribution - f1 energy")
plt.xlabel("z")
plt.ylabel("count")
plt.tight_layout()
plt.show()


###### 2.2 find_alpha

In [ ]:
def find_alpha(X_val, y_val,
               model_kriging,target_coverage,
               alpha_max=50, alpha_step=0.01):
  mean, std = model_kriging.predict(X_val)

  alpha = 0
  best_alpha = alpha_max
  while alpha < alpha_max:
      f_upper = mean + alpha * std
      coverage = np.mean(y_val <= f_upper)

      if coverage >= target_coverage:
            best_alpha = alpha
            print(f"coverage={coverage*100:.2f}%")
            break

      alpha += alpha_step

  return best_alpha

mean_f1, std_f1 = model_kriging_f1.predict(X_test_np)
alpha_c80_f1 = find_alpha(X_val_np, y_val_np_f1, model_kriging_f1, target_coverage=0.8)
print(f"alpha_c80_f1={alpha_c80_f1:.2f}")
F_upper_c80 = mean_f1 + alpha_c80_f1 * std_f1
print('F_upper_c80\n', F_upper_c80[0:5],'\n')

alpha_c90_f1 = find_alpha(X_val_np, y_val_np_f1, model_kriging_f1, target_coverage=0.9)
print(f"alpha_c90_f1={alpha_c90_f1:.2f}")
F_upper_c90 = mean_f1 + alpha_c90_f1 * std_f1
print('F_upper_c90\n', F_upper_c90[0:5],'\n')

alpha_c95_f1 = find_alpha(X_val_np, y_val_np_f1, model_kriging_f1, target_coverage=0.95)
print(f"alpha_c95_f1={alpha_c95_f1:.2f}")
F_upper_c95 = mean_f1 + alpha_c95_f1 * std_f1
print('F_upper_c95\n', F_upper_c95[0:5],'\n')

# f2 is deterministic PMV, so no predictive uncertainty is added to f2.
alpha_c80_f2 = 0.0
alpha_c90_f2 = 0.0
alpha_c95_f2 = 0.0


###### 3. Optimization

In [ ]:
# Algorithm
mutation = FixedWallMutation(prob=0.7, eta=5, fixed_wall_list=fixed_wall_list)
crossover = SBX(prob=0.8)

algorithm = NSGA2(
    pop_size=pop_size,
    crossover = crossover,
    mutation = mutation,
    survival=Survival_standard(),     ### change
    sampling=FromArraySampling(initial_population),
    eliminate_duplicates=True)

hv_list, time_list = [], []
solutions_list, objectives_list = [], []

flexible_problem_kriging = Flexible_Wall_Problem(
    model_f1=model_kriging_f1,
    n_var=n_var,
    n_obj=n_obj,
    xl=xl,
    xu=xu,
    fixed_wall_list=fixed_wall_list,
    room_types=room_types_list,
    required_num_meeting_room=required_num_meeting_room,
    space_width=space_width,
    room_area_min=room_area_min,
    initial_occ_list=initial_occ_list,
    total_occ=total_occ,
    occ_min_area=occ_min_area,
    th_zone_list=th_zone_list,
    outdoor_t=outdoor_t,
    outdoor_rh=outdoor_rh,
    wind_speed=wind_speed,
    solar_radiation=solar_radiation,
    clo=clo_default,
    met=met_default,
    v=v_default,
    use_surrogate='Kriging_uncertainty')

initial_F = flexible_problem_kriging.evaluate(initial_solution.reshape(1, -1), return_values_of=['F'])[0]
print(f"Initial f1={initial_F[0]:.4f}, f2={initial_F[1]:.4f}")

# Optimization
start_time = time.time()

res = minimize(
    flexible_problem_kriging,
    algorithm,
    termination = get_termination("n_gen", n_gen),
    seed=seed,
    save_history=True,
    verbose=True)

end_time = time.time()

solution = res.X
obj = res.F

hv_value = float(hv.do(obj))
hv_list.append(hv_value)
solutions_list.append(solution)
objectives_list.append(obj)

max_obj = np.max(obj, axis=0)
min_obj = np.min(obj, axis=0)
print(f"Seed {seed} | Time: {end_time - start_time:.2f}s | "
      f"HV: {hv_value:.4f} | "
      f"Min obj: {min_obj} | Max obj: {max_obj}")

time_list.append(end_time - start_time)

solution_standard = solutions_list[-1]
obj_standard = objectives_list[-1]
plot_obj_2d(obj_standard, xlim=(0, ref_point[0]), ylim=(0, ref_point[1]))


In [ ]:
mean_hv, std_hv = mean_std(hv_list)
mean_time, std_time = mean_std(time_list)

print('Problem name: ', problem_name)
print("\n=== GPR (RBF) ===")
print(f"HV: Mean = {mean_hv:.4f}, Std = {std_hv:.4f}")
print(f"Time: Mean = {mean_time:.2f}, Std = {std_time:.2f}")

results_standard = pd.DataFrame(obj_standard, columns=['f1_cost', 'f2_thermal_comfort'])
for i in range(solution_standard.shape[1]):
    results_standard[f'x{i}'] = solution_standard[:, i]

results_standard.sort_values('f1_cost').head(10)


###### 3. Optimization Kriging (RBF) + dual-ranking (c=0.90)

In [ ]:
# Algorithm
algorithm = NSGA2(
    pop_size=pop_size,
    crossover = crossover,
    mutation = mutation,
    survival=Survival_dual_ranking(alpha_f1=alpha_c90_f1,
                                   alpha_f2=alpha_c90_f2),     ### change
    sampling=FromArraySampling(initial_population),
    eliminate_duplicates=True)

hv_list, time_list = [], []
solutions_list, objectives_list = [], []

flexible_problem = Flexible_Wall_Problem(
    model_f1=model_kriging_f1,   ### change
    n_var=n_var,
    n_obj=n_obj,
    xl=xl,
    xu=xu,
    fixed_wall_list=fixed_wall_list,
    room_types=room_types_list,
    required_num_meeting_room=required_num_meeting_room,
    space_width=space_width,
    room_area_min=room_area_min,
    initial_occ_list=initial_occ_list,
    total_occ=total_occ,
    occ_min_area=occ_min_area,
    th_zone_list=th_zone_list,
    outdoor_t=outdoor_t,
    outdoor_rh=outdoor_rh,
    wind_speed=wind_speed,
    solar_radiation=solar_radiation,
    clo=clo_default,
    met=met_default,
    v=v_default,
    use_surrogate='Kriging_uncertainty')   ### change

initial_F = flexible_problem.evaluate(initial_solution.reshape(1, -1), return_values_of=['F'])[0]
print(f"Initial f1={initial_F[0]:.4f}, f2={initial_F[1]:.4f}")

# Optimization
start_time = time.time()

res = minimize(
    flexible_problem,
    algorithm,
    termination = get_termination("n_gen", n_gen),
    seed=seed,
    save_history=True,
    verbose=True)

end_time = time.time()

solution = res.X
obj = res.F

hv_value = float(hv.do(obj))
hv_list.append(hv_value)
solutions_list.append(solution)
objectives_list.append(obj)

max_obj = np.max(obj, axis=0)
min_obj = np.min(obj, axis=0)
print(f"Seed {seed} | Time: {end_time - start_time:.2f}s | "
      f"HV: {hv_value:.3f} | "
      f"Min obj: {min_obj} | Max obj: {max_obj}")

time_list.append(end_time - start_time)

solution_dr = solutions_list[-1]
obj_dr = objectives_list[-1]
plot_obj_2d(obj_dr, xlim=(0, ref_point[0]), ylim=(0, ref_point[1]))


In [ ]:
mean_hv, std_hv = mean_std(hv_list)
mean_time, std_time = mean_std(time_list)

print('Problem name: ', problem_name)
print("\n=== GPR (RBF) + c90===")
print(f"HV: Mean = {mean_hv:.4f}, Std = {std_hv:.4f}")
print(f"Time: Mean = {mean_time:.2f}, Std = {std_time:.2f}")

results_dr = pd.DataFrame(obj_dr, columns=['f1_cost', 'f2_thermal_comfort'])
for i in range(solution_dr.shape[1]):
    results_dr[f'x{i}'] = solution_dr[:, i]

OUTPUT_DIR = PROJECT_DIR / 'results_gpr_rbf_like_test'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results_standard.to_csv(OUTPUT_DIR / 'standard_solutions.csv', index=False)
results_dr.to_csv(OUTPUT_DIR / 'dual_ranking_c90_solutions.csv', index=False)
print('Saved:', OUTPUT_DIR)

results_dr.sort_values('f1_cost').head(10)


In [ ]:
print(f"obj_standard HV: {hv.do(obj_standard):.4f}")
print(f"obj_dr HV: {hv.do(obj_dr):.4f}")

In [ ]:
# Plot only nondominated solutions for paper
def plot_exp(F, title=None, fig_name=None, f1=None, f2=None,
             color="#87CEEB", xlim=None, ylim=None):
    n_obj = F.shape[1]

    if n_obj == 2:
        nds = NonDominatedSorting()
        front_idx = nds.do(F, only_non_dominated_front=True)

        pareto_F = F[front_idx]

        fig, ax = plt.subplots(figsize=(7, 6))

        # only plot nondominated solutions
        ax.scatter(pareto_F[:, 0], pareto_F[:, 1], s=50, color=color)

        if xlim is not None:
            ax.set_xlim(xlim)
        if ylim is not None:
            ax.set_ylim(ylim)

        if f1 == 'f1':
            ax.set_xlabel("$f_1$", fontsize=30)
        if f2 == 'f2':
            ax.set_ylabel("$f_2$", fontsize=30)

        ax.tick_params(direction='out', length=5, width=1, colors='black')
        ax.tick_params(labelsize=20)

        if title:
            ax.set_title(title, fontsize=25)

        fig.tight_layout()

        if fig_name is not None:
            plt.savefig(f"{fig_name}.pdf", dpi=300, bbox_inches='tight')

        plt.show()


# ===== first find global min/max from both inputs =====
all_F = np.vstack([obj_standard, obj_dr])

x_min, x_max = all_F[:, 0].min(), all_F[:, 0].max()
y_min, y_max = all_F[:, 1].min(), all_F[:, 1].max()

x_margin = 0.05 * (x_max - x_min)
y_margin = 0.05 * (y_max - y_min)

xlim = (x_min - x_margin, x_max + x_margin)
ylim = (y_min - y_margin, y_max + y_margin)


plot_exp(obj_standard,
         fig_name='fig_GPR_RBF',
         color="#FFA500",
         xlim=xlim,
         ylim=ylim)

plot_exp(obj_dr,
         fig_name='fig_GPR_RBF_DR',
         color="#FFA500",
         xlim=xlim,
         ylim=ylim)